# This will be our project

Before running the notebook, set DATA_DIR in the first cell to point to your local dataset folder.

Below the code for K-means clustering, PCA, and necessary preprocessing.

To start, download the dataset and unzip it to a folder named data/ in the Project3 directory. Here is the link: https://drive.google.com/file/d/1Z8IdHF6iOaJ5Ju3LMlaQQGKq4GCqc79b/view?usp=sharing

Contains resampled cells size 32x32x3
Split:

80% train:
    416 dead
    - 1131 live

10% dev:
    52 dead
    - 141 live

10% test:
    52 dead
    - 142 live

In [4]:
import os
import numpy as np
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# DATA_DIR = r"C:\\Users\\elmor\\Documents\\Oslo_sem1\\Machine_Learning\\cells\\cells"
DATA_DIR = r"../data/cells"

def load_split(split):
    images = []
    labels = []

    split_dir = os.path.join(DATA_DIR, split)
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"Split directory not found: {split_dir}")

    class_map = {"live": "Live_resized", "dead": "Dead_resized"}
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}

    for label in ["live", "dead"]:
        folder = os.path.join(split_dir, class_map[label])
        if not os.path.isdir(folder):
            raise FileNotFoundError(f"Class folder not found: {folder}")
        files = [f for f in os.listdir(folder)
                 if os.path.isfile(os.path.join(folder, f))
                 and os.path.splitext(f)[1].lower() in valid_exts]
        for fname in sorted(files):
            path = os.path.join(folder, fname)
            img = Image.open(path).convert("RGB")
            if img.size != (32, 32):
                img = img.resize((32, 32))
            images.append(np.array(img))
            labels.append(0 if label == "live" else 1)
    return np.array(images), np.array(labels)

X_train, y_train = load_split("train")
X_dev, y_dev = load_split("dev")
X_test, y_test = load_split("test")

print("Loaded shapes:", X_train.shape, X_dev.shape, X_test.shape)

Loaded shapes: (1547, 32, 32, 3) (193, 32, 32, 3) (194, 32, 32, 3)


In [5]:
# Quick verification: per-split class counts (0=live, 1=dead)
import numpy as np
def _counts(y):
    return {0: int(np.sum(y == 0)), 1: int(np.sum(y == 1))}
print("Train counts:", _counts(y_train))
print("Dev counts:", _counts(y_dev))
print("Test counts:", _counts(y_test))

Train counts: {0: 1131, 1: 416}
Dev counts: {0: 141, 1: 52}
Test counts: {0: 142, 1: 52}


## Preprocessing for PCA
First flattening the images 32x32x3 to 3072-dimensional vectors

In [ ]:
def flatten_images(X):
    return X.reshape(len(X), -1)

X_train_flat = flatten_images(X_train)
X_dev_flat = flatten_images(X_dev)
X_test_flat = flatten_images(X_test)

print("Flattened shapes:", X_train_flat.shape, X_dev_flat.shape, X_test_flat.shape)


Normalizing pixels from [0,255] (RGB) to [0,1]

In [ ]:
X_train_flat = X_train_flat.astype(np.float32) / 255.0
X_dev_flat = X_dev_flat.astype(np.float32) / 255.0
X_test_flat = X_test_flat.astype(np.float32) / 255.0

## PCA, reducing the dimensionality
PCA loop for [2, 5, 10, 20, 50, 100]: principal components

In [ ]:
for m in [2, 5, 10, 20, 50, 100]:
    print(f"\nPCA with {m} components")
    pca = PCA(n_components=m)
    Z_train = pca.fit_transform(X_train_flat)
    Z_dev = pca.transform(X_dev_flat)
    Z_test = pca.transform(X_test_flat)

    print("explained variance ratio:", pca.explained_variance_ratio_.sum())
    print("Shape:", Z_train.shape)